In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from frag_gnn.runner import load_config, init_dataset, init_dataloader
from frag_gnn.pl_model import FragGNNPL
from frag_gnn.dataset import SpecMolFragDataset
from frag_gnn.utils.misc_utils import scatter_reduce, scatter_argmax, scatter_argtopk, EPS, booltype, scatter_masked_softmax, scatter_logsumexp, safelog, LOG_ZERO, flatten_lol
from frag_gnn.utils.script_utils import init_wandb_ckpt, run_inference, select_model_vals, sparse_kl, log_mean, log_std, pearson_r
from frag_gnn.loss import sparse_conditional_entropy_fn, sparse_cosine_distance_hungarian
from frag_gnn.utils.frag_utils import load_frag_d
from frag_gnn.graff.dataset import calculate_prec_formula
from frag_gnn.graff.data_utils import fit_ann_library
from frag_gnn.utils.data_utils import composition_to_string
from frag_gnn.utils.spec_utils import calculate_match_mzs

import numpy as np
import torch as th
import pandas as pd
import argparse	
import wandb
import os
from pprint import pprint
from tqdm import tqdm
from pyteomics.mass import Composition

In [3]:
# load ann_df
# get formulae from products
proc_dp = "../data/proc/nist20mona23v3"
spec_df = pd.read_pickle(f"{proc_dp}/spec_df.pkl")
mol_df = pd.read_pickle(f"{proc_dp}/mol_df.pkl")
ann_df = pd.read_pickle(f"{proc_dp}/ann_df.pkl")

In [4]:
print(ann_df.columns)

Index(['dset_spec_id', 'mol_id', 'dset', 'spec_id', 'prec_type', 'formula',
       'ann_peak_mzs', 'ann_products', 'ann_losses', 'ann_isotopes',
       'ann_exact_mzs'],
      dtype='object')


In [5]:
# merge ann_df
m_ann_df = ann_df[["spec_id","ann_products","ann_exact_mzs"]].copy()
m_ann_df = m_ann_df.merge(spec_df[["spec_id","group_id"]], on="spec_id", how="inner")
m_ann_df = m_ann_df.drop(columns="spec_id").groupby("group_id").agg({"ann_products":list, "ann_exact_mzs":list}).reset_index()

In [6]:
# dedup formulae
def formula_dedup_fn(row):
    formulae = flatten_lol(row["ann_products"])
    mzs = flatten_lol(row["ann_exact_mzs"])
    assert len(formulae) == len(mzs), (len(formulae), len(mzs))
    formulae_dedup = []
    mzs_dedup = []
    for i in range(len(formulae)):
        formula = formulae[i]
        mz = mzs[i]
        if formula not in formulae_dedup:
            assert mz not in mzs_dedup, (i, mz, mzs_dedup)
            formulae_dedup.append(formula)
            mzs_dedup.append(mz)
    # assert len(formulae_dedup) == len(set(formulae_dedup)), (len(formulae_dedup), len(set(formulae_dedup)))
    out_d = {
        "group_id": row["group_id"],
        "formulae_dedup": formulae_dedup,
        "mzs_dedup": mzs_dedup
    }
    return out_d

m_ann_df = m_ann_df.apply(formula_dedup_fn, axis=1, result_type="expand")

In [7]:
print(ann_df["spec_id"].nunique())
print(m_ann_df["group_id"].nunique())
print(m_ann_df.columns)

397057
60885
Index(['group_id', 'formulae_dedup', 'mzs_dedup'], dtype='object')


In [8]:
baseline_vals_fp = "../inference_outputs/entropyv3/scaffold_d4_baseline_test_False/vals.pkl"
baseline_vals = th.load(baseline_vals_fp)

In [9]:
print(baseline_vals.keys())
print(baseline_vals["entropyv3_scaffold_d4_baseline_ens_s00"].keys())

dict_keys(['entropyv3_scaffold_d4_baseline_ens_s14', 'entropyv3_scaffold_d4_baseline_ens_s12', 'entropyv3_scaffold_d4_baseline_ens_s11', 'entropyv3_scaffold_d4_baseline_ens_s10', 'entropyv3_scaffold_d4_baseline_ens_s09', 'entropyv3_scaffold_d4_baseline_ens_s08', 'entropyv3_scaffold_d4_baseline_ens_s07', 'entropyv3_scaffold_d4_baseline_ens_s00', 'entropyv3_scaffold_d4_baseline_ens_s13', 'entropyv3_scaffold_d4_baseline_ens_s05', 'entropyv3_scaffold_d4_baseline_ens_s06', 'entropyv3_scaffold_d4_baseline_ens_s03', 'entropyv3_scaffold_d4_baseline_ens_s04', 'entropyv3_scaffold_d4_baseline_ens_s02', 'entropyv3_scaffold_d4_baseline_ens_s01'])
dict_keys(['unique_id', 'smiles', 'mean_loss', 'total_loss', 'total_weight', 'spec_per_group', 'spec_per_mol', 'group_per_mol', 'true_mzs', 'true_logprobs', 'true_batch_idxs', 'true_prec_mzs', 'pred_mzs', 'pred_logprobs', 'pred_batch_idxs', 'pred_formula_logprobs', 'pred_formula_formula_idxs', 'pred_formula_batch_idxs', 'pred_node_logprobs', 'pred_node_nod

In [10]:
def get_formula_vals(vals):

    select_keys = [
        "pred_formula_logprobs",
        "pred_formula_batch_idxs",
        "pred_formula_formula_idxs",
        "unique_id",
        "pred_mzs",
        "pred_logprobs",
        "pred_batch_idxs",
        "true_mzs",
        "true_logprobs",
        "true_batch_idxs",
    ]

    select_d = select_model_vals(vals,select_keys,stack_dim=0)

    return select_d

In [11]:
# load pred_mzs, pred_logprobs, batch_idxs
# map pred_mzs to pred_formula_idxs
formula_vals = get_formula_vals(
    {
        "entropyv3_scaffold_d4_baseline_ens_s00": baseline_vals["entropyv3_scaffold_d4_baseline_ens_s00"]
    }
)


In [12]:
# get molecule ids
group_ids = formula_vals["unique_id"].flatten().numpy()
id_df = spec_df[["spec_id","mol_id","group_id"]]
id_df = id_df[id_df["group_id"].isin(group_ids)]
group_mol_df = id_df[["group_id","mol_id"]].drop_duplicates()
mol_ids = group_mol_df.set_index("group_id").loc[group_ids,"mol_id"].values

In [13]:
print(len(mol_ids))

3341


In [14]:
# use molecule id to load frag_d
mol_ids_str = [str(mol_id) for mol_id in mol_ids]
frag_d_d = {}
frag_dp = "../data/frag/nist20mona23v3_d4_h4_isoFalse_nb3/dags"
for mol_id_str in tqdm(mol_ids_str):
    frag_d_d[int(mol_id_str)] = load_frag_d(mol_id_str, frag_dp, is_compressed=True)

  0%|          | 0/3341 [00:00<?, ?it/s]

100%|██████████| 3341/3341 [02:23<00:00, 23.31it/s]


In [15]:
np.isin(group_ids,m_ann_df["group_id"]).mean()

0.9865309787488776

In [22]:
unique_ids = formula_vals["unique_id"].squeeze(0)
true_batch_idxs = formula_vals["true_batch_idxs"].squeeze(0)
pred_batch_idxs = formula_vals["pred_batch_idxs"].squeeze(0)
true_mzs = formula_vals["true_mzs"].squeeze(0)
true_logprobs = formula_vals["true_logprobs"].squeeze(0)
pred_mzs = formula_vals["pred_mzs"].squeeze(0)
pred_logprobs = formula_vals["pred_logprobs"].squeeze(0)
spec_batch_idxs = th.arange(unique_ids.shape[0])
pred_formula_batch_idxs = formula_vals["pred_formula_batch_idxs"].squeeze(0)
pred_formula_formula_idxs = formula_vals["pred_formula_formula_idxs"].squeeze(0)
formula_idx_cumsum = th.cumsum(th.cat(
    [
        th.zeros([1],dtype=th.long),
        th.unique(pred_formula_batch_idxs, return_counts=True)[1]
    ], dim=0), dim=0
)

m_ann_df_cp = m_ann_df.copy().set_index("group_id")

ann_formulae, pred_formulae = [], []
coverages, peak_coverages, w_peak_coverages = [], [], []

# iterate over the group truth data
for i in tqdm(range(len(group_ids))):
    i_group_id = group_ids[i]
    i_mol_id = mol_ids[i]
    if i_group_id not in m_ann_df_cp.index:
        continue
    i_frag_d = frag_d_d[i_mol_id]
    i_formula_idx_to_str = i_frag_d["idx_to_formula"]
    # pprint(i_formula_idx_to_str)
    i_mask = (unique_ids == i_group_id).flatten()
    i_batch_idx = spec_batch_idxs[i_mask].item()
    i_pred_peak_mask = (pred_batch_idxs == i_batch_idx)
    i_pred_mzs = pred_mzs[i_pred_peak_mask]
    # i_pred_logprobs = pred_logprobs[i_pred_peak_mask]
    i_true_peak_mask = (true_batch_idxs == i_batch_idx)
    i_true_mzs = true_mzs[i_true_peak_mask]
    i_true_logprobs = true_logprobs[i_true_peak_mask]
    assert th.isclose(th.logsumexp(i_true_logprobs,dim=0),th.tensor(0.),rtol=0.,atol=1e-5), th.logsumexp(i_true_logprobs,dim=0)
    i_formula_mask = (pred_formula_batch_idxs == i_batch_idx).flatten()
    i_pred_formula_idxs = pred_formula_formula_idxs[i_formula_mask] - formula_idx_cumsum[i_batch_idx]
    assert i_pred_formula_idxs[0] == 0, i_pred_formula_idxs
    # TODO: modify to account for prec_type properly
    i_pred_formulae = [
        composition_to_string(Composition(formula=i_formula_idx_to_str[idx.item()]) + Composition(formula="H")) 
        for idx in i_pred_formula_idxs[1:]
    ]
    i_pred_formulae = np.array(i_pred_formulae)
    i_ann_formulae = m_ann_df_cp.loc[i_group_id,"formulae_dedup"]
    i_ann_formulae = [
        composition_to_string(Composition(formula=formula)) for formula in i_ann_formulae
    ]
    i_ann_formulae = np.array(i_ann_formulae)
    i_coverage = len(np.intersect1d(i_pred_formulae,i_ann_formulae)) / len(np.unique(i_ann_formulae))
    i_ann_mzs = th.tensor(m_ann_df_cp.loc[i_group_id,"mzs_dedup"])
    # match formulae within 10 ppm
    i_ann_mz_match_mask = calculate_match_mzs(
        i_ann_mzs, 
        i_true_mzs,
        tolerance=1e-5, # 10 ppm
        relative=True
    ) # A x T
    i_pred_mz_match_mask = calculate_match_mzs(
        i_pred_mzs, 
        i_true_mzs,
        tolerance=1e-5, # 10 ppm
        relative=True
    ) # P x T
    i_both_formula_match_mask = th.as_tensor(
        i_ann_formulae.reshape(-1,1) == i_pred_formulae.reshape(1,-1)
    ) # A x P
    i_ann_mz_match_mask_2 = (i_both_formula_match_mask.long() @ i_pred_mz_match_mask.long()) > 0 # A x T
    i_peak_match = th.any(i_ann_mz_match_mask & i_ann_mz_match_mask_2, dim=0)
    i_peak_coverage = i_peak_match.float().sum() / th.any(i_ann_mz_match_mask, dim=0).sum()
    i_wpeak_coverage = (i_peak_match.float() * i_true_logprobs.exp()).sum() / i_true_logprobs.exp().sum()

    pred_formulae.append(i_pred_formulae)
    ann_formulae.append(i_ann_formulae)
    coverages.append(i_coverage)
    peak_coverages.append(i_peak_coverage)
    w_peak_coverages.append(i_wpeak_coverage)

print(np.mean(coverages), np.min(coverages), np.max(coverages))
print(np.mean(peak_coverages), np.min(peak_coverages), np.max(peak_coverages))
print(np.mean(w_peak_coverages), np.min(w_peak_coverages), np.max(w_peak_coverages))

100%|██████████| 3341/3341 [20:13<00:00,  2.75it/s]   

0.7515012221493163 0.16593886462882096 1.0
0.8093528 0.16129032 1.0
0.8857702 0.12281597 1.0


In [ ]:
# get pred formula probabilities and strings
# get pred formula probailities
# filter formulae with prob > 0.01
# get true peak probabilities and mzs
# match true peak probabilities to ann formulae with ann mzs
# filter peaks with true intensity > 0.01
# calculate agreement between ann formulae and pred formulae, based on strings

In [ ]:
# two ways of evaluating the "formula accuracy"
# first way: what % of formulae produced by NIST are in the set of formulae that can be explained by our model?
# second way: calculate a "ground-truth" formula distribution for NIST, then compare cosine similarity/JSD with ours